## 1. Import Libraries

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from tqdm.notebook import tqdm
import time
import re
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Load Bank Data

In [4]:
# Load data bank syariah dari CSV
df_banks = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_syariah.csv')

print(f"Total banks loaded: {len(df_banks)}")
print(f"Provinsi available: {len(df_banks['Provinsi'].unique())} provinsi")
df_banks.head(10)

Total banks loaded: 194
Provinsi available: 25 provinsi


,Provinsi,Kabupaten/Kota,Nama Bank,Kode Bank
0,Provinsi Jawa Barat,Kab. Bekasi,PT Bank Perekonomian Rakyat Syariah Amanah Insani,620066
1,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Amanah Ummah,620004
2,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Botani Bin...,620061
3,Provinsi Jawa Barat,Kab. Bogor,PT. BPRS Rif'atul Ummah,620068
4,Provinsi Jawa Barat,Kab. Bogor,PT Bank Perekonomian Rakyat Syariah Harta Insa...,620069
5,Provinsi Jawa Barat,Kab. Bogor,PT BPRS Bogor Tegar Beriman,620177
6,Provinsi Jawa Barat,Kab. Cianjur,PT Bank Perekonomian Rakyat Syariah Gaido Indo...,620043
7,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Amanah Rab...,620002
8,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Almasoem,620027
9,Provinsi Jawa Barat,Kab. Bandung,PT Bank Perekonomian Rakyat Syariah Al Ihsan,620044


## 3. Configuration

In [5]:
BASE_URL = "https://cfs.ojk.go.id/cfs/ReportViewerForm.aspx"
MONTH = "12"  # Fixed: Desember
PERIOD_TYPE = "R"

# Report types untuk BPRS (2019-2024)
REPORT_TYPES = {
    "BPS-901-000001": "Laporan Posisi Keuangan Publikasi",
    "BPS-901-000002": "Laporan Laba Rugi Publikasi",
    "BPS-901-000003": "Rasio Keuangan"
}

# Column order based on Dataset Syakhsan.xlsx - BPRS (2019 - 2024)
COLUMN_ORDER = [
    'Tahun', 'Bulan', 'Longitude', 'Latitude', 'Nama_BPR', 
    'Kabupaten_Kota', 'Provinsi', 'Kode_Bank',
    # Laporan Posisi Keuangan
    'Total_Aset',
    'Piutang_Murabahah',
    'Piutang_Istishna',
    'Piutang_Multijasa',
    'Piutang_Qardh',
    'Piutang_Sewa',
    'Pembiayaan_Mudharabah',
    'Pembiayaan_Musyarakah',
    'Pembiayaan_Lainnya',
    'Salam',
    'Agunan_yang_Diambil_Alih',
    'Tabungan_Wadiah',
    'Tabungan_Mudharabah',
    'Deposito_Mudharabah',
    'Cadangan_Kerugian_Penurunan_Nilai_1',
    'Cadangan_Kerugian_Penurunan_Nilai_2',
    'Liabilitas_Segera',
    'Liabilitas_kepada_BI',
    'Liabilitas_kepada_Bank_Lain',
    'Pembiayaan_Diterima',
    'Liabilitas_Lainnya',
    'Dana_Syirkah_Temporer',
    # Laporan Laba Rugi
    'Pendapatan_setelah_distribusi_bagi_hasil',
    'Pendapatan_dari_penyaluran_dana',
    'Beban_Operasional',
    'Zakat',
    'Laba_Rugi_Bersih',
    # Rasio Keuangan
    'KPMM',
    'NPF_Neto',
    'NPF_Gross',
    'ROA',
    'BOPO',
    'FDR',
    'Cash_Ratio',
    'NI'
]

print("✓ Configuration loaded")

✓ Configuration loaded


## 4. Helper Functions

In [6]:
def clean_number(value):
    """Convert string number dengan format Indonesia ke float"""
    if pd.isna(value) or value == '' or value == 'NaN':
        return 0
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value = str(value).strip()
    value = re.sub(r'[^\d.,()\-]', '', value)
    
    if '(' in value and ')' in value:
        value = '-' + value.replace('(', '').replace(')', '')
    
    value = value.replace(',', '')
    
    try:
        return float(value)
    except:
        return 0

def find_value_in_row(df, keywords, col_index=2, return_zero_if_not_found=True):
    """
    Cari nilai di tabel berdasarkan keyword
    return_zero_if_not_found: Jika False, return None jika tidak ketemu (untuk bedakan dari nilai 0 actual)
    """
    for col in df.columns[:2]:
        for keyword in keywords:
            mask = df[col].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            if mask.any():
                idx = df[mask].index[0]
                if col_index < len(df.columns):
                    return clean_number(df.iloc[idx, col_index])
                return 0
    
    # Jika tidak ketemu sama sekali
    return 0 if return_zero_if_not_found else None

print("Helper functions defined")

Helper functions defined


## 5. Extraction & Parsing Functions

In [7]:
def extract_table(bank_code_number, bank_code, year, report_type, table_index=16, timeout=15):
    """Ekstrak tabel dari laporan BPRS"""
    params = {
        'BankCodeNumber': bank_code_number,
        'BankCode': bank_code,
        'Month': MONTH,
        'Year': str(year),
        'FinancialReportPeriodTypeCode': PERIOD_TYPE,
        'FinancialReportTypeCode': report_type
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        
        # Extract tables
        dfs = pd.read_html(StringIO(response.text))
        
        if not dfs or table_index >= len(dfs):
            return None
        
        return dfs[table_index]
        
    except Exception as e:
        return None

def parse_posisi_keuangan(df):
    """Parse Laporan Posisi Keuangan BPRS"""
    data = {}
    
    # Mapping fields - Updated dengan keyword yang lebih spesifik
    fields_mapping = {
        'Total_Aset': [['Total Aset', 'JUMLAH ASET'], 2],
        'Piutang_Murabahah': [['a. Piutang Murabahah', 'Piutang Murabahah'], 2],
        'Piutang_Istishna': [['b. Piutang Istishna', 'Piutang Istishna'], 2],
        'Piutang_Multijasa': [['c. Piutang Multijasa', 'Piutang Multijasa'], 2],
        'Piutang_Qardh': [['d. Piutang Qardh', 'Piutang Qardh'], 2],
        'Piutang_Sewa': [['e. Piutang Sewa', 'Piutang Sewa'], 2],
        'Pembiayaan_Mudharabah': [['a. Mudharabah', 'Mudharabah'], 2],
        'Pembiayaan_Musyarakah': [['b. Musyarakah', 'Musyarakah'], 2],
        'Pembiayaan_Lainnya': [['c. Lainnya'], 2],
        'Salam': [['9. Salam', '8. Salam', 'Salam'], 2],
        'Agunan_yang_Diambil_Alih': [['12. Agunan yang Diambil Alih', 'Agunan yang Diambil Alih'], 2],
        'Tabungan_Wadiah': [['2. Tabungan Wadiah', 'Tabungan Wadiah'], 2],
        'Tabungan_Mudharabah': [['a. Tabungan', 'Tabungan Mudharabah'], 2],
        'Deposito_Mudharabah': [['b. Deposito', 'Deposito Mudharabah'], 2],
        'Liabilitas_Segera': [['1. Liabilitas Segera', 'Liabilitas Segera'], 2],
        'Liabilitas_kepada_BI': [['4. Liabilitas kepada Bank Indonesia', 'Liabilitas kepada Bank Indonesia'], 2],
        'Liabilitas_kepada_Bank_Lain': [['5. Liabilitas kepada Bank Lain', 'Liabilitas kepada Bank Lain'], 2],
        'Pembiayaan_Diterima': [['6. Pembiayaan Diterima', 'Pembiayaan Diterima'], 2],
        'Liabilitas_Lainnya': [['7. Liabilitas Lainnya', 'Liabilitas Lainnya'], 2],
        'Dana_Syirkah_Temporer': [['8. Dana Investasi Profit Sharing', 'Dana Syirkah Temporer', 'Dana Investasi Profit Sharing'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    # Special handling untuk Cadangan Kerugian Penurunan Nilai
    # Ada 2 jenis: Umum dan Khusus - kita ambil keduanya lalu jumlahkan
    cadangan_umum = find_value_in_row(df, ['a. Umum', 'Penyisihan Penghapusan Aset Produktif'], 2)
    cadangan_khusus = find_value_in_row(df, ['b. Khusus', '16. Cadangan Kerugian Penurunan Nilai -/-', 'Cadangan Kerugian Penurunan Nilai -/-'], 2)
    
    data['Cadangan_Kerugian_Penurunan_Nilai_1'] = cadangan_umum
    data['Cadangan_Kerugian_Penurunan_Nilai_2'] = cadangan_khusus
    
    return data

def parse_laba_rugi(df):
    """Parse Laporan Laba Rugi BPRS"""
    data = {}
    
    fields_mapping = {
        'Pendapatan_setelah_distribusi_bagi_hasil': [['III. Pendapatan setelah distribusi bagi hasil', 'Pendapatan setelah distribusi'], 2],
        'Pendapatan_dari_penyaluran_dana': [['I. Pendapatan Dari Penyaluran Dana', 'Pendapatan Dari Penyaluran Dana'], 2],
        'Beban_Operasional': [['V. Beban Operasional', 'Beban Operasional'], 2],
        'Zakat': [['X. Zakat'], 2],
        'Laba_Rugi_Bersih': [['XI. Laba Rugi Bersih', 'Laba Rugi Bersih', 'XII. Laba Rugi Bersih'], 2],
    }
    
    for field, (keywords, col_idx) in fields_mapping.items():
        value = find_value_in_row(df, keywords, col_idx)
        data[field] = value
    
    return data

def parse_rasio_keuangan(df):
    """Parse Rasio Keuangan BPRS"""
    data = {}
    
    # Rasio keywords
    ratio_mappings = {
        'KPMM': ['Kewajiban Penyediaan Modal Minimum', 'KPMM'],
        'NPF_Neto': ['Non Performing Financing (NPF) Neto', 'NPF) Neto', 'NPF Neto', 'NPF Net'],
        'NPF_Gross': ['Non Perfoming Financing (NPF) Gross', 'NPF) Gross', 'NPF Gross'],
        'ROA': ['Return on Asset', 'ROA'],
        'BOPO': ['Beban Operasional terhadap Pendapatan Operasional', 'BOPO'],
        'FDR': ['Financing to Deposit Ratio', 'FDR'],
        'Cash_Ratio': ['Cash Ratio'],
        'NI': ['Net Imbalan', 'NI']
    }
    
    for field_name, keywords in ratio_mappings.items():
        found = False
        for row_idx in range(len(df)):
            row_text = ' '.join([str(df.iloc[row_idx, col]) for col in range(len(df.columns))]).lower()
            
            for keyword in keywords:
                if keyword.lower() in row_text:
                    # Get last column value (ratio column)
                    for col_idx in range(len(df.columns) - 1, -1, -1):
                        val_str = str(df.iloc[row_idx, col_idx])
                        if val_str not in ['NaN', 'nan', ''] and any(c.isdigit() for c in val_str):
                            val = clean_number(val_str)
                            # Ratio values should be between -100 to 1000%
                            if -100 < val < 1000:
                                data[field_name] = val
                                found = True
                                break
                    if found:
                        break
            if found:
                break
    
    # Fill missing ratios
    for field in ['KPMM', 'NPF_Neto', 'NPF_Gross', 'ROA', 'BOPO', 'FDR', 'Cash_Ratio', 'NI']:
        if field not in data:
            data[field] = 0
    
    return data

print("✓ Extraction & Parsing functions defined")

✓ Extraction & Parsing functions defined


## 6. Main Scraper Class

In [8]:
class BPRSScraper:
    """Main scraper class untuk BPR Syariah"""
    
    def __init__(self):
        self.results = []
        
    def scrape_single_bank(self, bank_row, year):
        """Scrape data untuk satu bank di satu tahun"""
        bank_code_number = str(bank_row['Kode Bank'])
        bank_code = bank_row['Nama Bank']
        
        result = {
            'Tahun': int(year),
            'Bulan': MONTH,
            'Nama_BPR': bank_code,
            'Kabupaten_Kota': bank_row['Kabupaten/Kota'],
            'Provinsi': bank_row['Provinsi'],
            'Kode_Bank': bank_code_number,
            'Longitude': None,
            'Latitude': None,
            'status': 'failed'
        }
        
        try:
            # Laporan Posisi Keuangan (table index 16)
            df_posisi = extract_table(bank_code_number, bank_code, year, "BPS-901-000001", table_index=16)
            
            if df_posisi is not None:
                posisi_data = parse_posisi_keuangan(df_posisi)
                result.update(posisi_data)
            
            time.sleep(0.1)
            
            # Laporan Laba Rugi (table index 16)
            df_laba = extract_table(bank_code_number, bank_code, year, "BPS-901-000002", table_index=16)
            
            if df_laba is not None:
                laba_data = parse_laba_rugi(df_laba)
                result.update(laba_data)
            
            time.sleep(0.1)
            
            # Rasio Keuangan (table index 16)
            df_rasio = extract_table(bank_code_number, bank_code, year, "BPS-901-000003", table_index=16)
            
            if df_rasio is not None:
                rasio_data = parse_rasio_keuangan(df_rasio)
                result.update(rasio_data)
            
            result['status'] = 'success'
            
        except Exception as e:
            result['error'] = str(e)
        
        return result
    
    def run(self, df_banks_filtered, years, max_workers=4):
        """Run scraping dengan ThreadPoolExecutor"""
        tasks = []
        for _, bank_row in df_banks_filtered.iterrows():
            for year in years:
                tasks.append((bank_row, year))
        
        total_tasks = len(tasks)
        results = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(self.scrape_single_bank, task[0], task[1]): task for task in tasks}
            
            with tqdm(total=total_tasks, desc="Scraping", unit="task") as pbar:
                for future in as_completed(futures):
                    try:
                        result = future.result()
                        results.append(result)
                    except Exception as e:
                        print(f"Task failed: {e}")
                    pbar.update(1)
        
        return results

print("BPRSScraper class defined")

BPRSScraper class defined


## 7. USER CONFIGURATION

**Edit parameter di bawah ini:**

In [9]:
# ============================================================================
# EDIT PARAMETER DI SINI
# ============================================================================

# Rentang tahun (2019-2024)
YEAR_START = 2024
YEAR_END = 2024

# Pilih provinsi:
# - ['SEMUA'] untuk scrape semua provinsi
# - Atau list provinsi spesifik: ['Provinsi Jawa Barat', 'Provinsi DKI Jakarta']
SELECTED_PROVINCES = ['SEMUA']

# Jumlah thread workers (4-8 optimal)
MAX_WORKERS = 8

# Nama file output
OUTPUT_FILENAME = 'bprs_financial_data'

# ============================================================================

print("Configuration set:")
print(f"   Tahun: {YEAR_START} - {YEAR_END}")
print(f"   Provinsi: {SELECTED_PROVINCES}")
print(f"   Workers: {MAX_WORKERS}")

Configuration set:
   Tahun: 2024 - 2024
   Provinsi: ['SEMUA']
   Workers: 8


## 8. RUN SCRAPING

**Jalankan cell ini untuk memulai scraping:**

In [10]:
# Prepare data
years = list(range(YEAR_START, YEAR_END + 1))

# Filter banks by province
if 'SEMUA' in SELECTED_PROVINCES:
    filtered_banks = df_banks.copy()
else:
    filtered_banks = df_banks[df_banks['Provinsi'].isin(SELECTED_PROVINCES)].copy()

total_tasks = len(filtered_banks) * len(years)

print("="*70)
print(" STARTING SCRAPING - BPR SYARIAH")
print("="*70)
print(f"Tahun: {YEAR_START} - {YEAR_END} ({len(years)} tahun)")
print(f"Bulan: Desember (Fixed)")
print(f"Provinsi: {', '.join(SELECTED_PROVINCES)}")
print(f"Jumlah Bank: {len(filtered_banks)}")
print(f"Total Tasks: {total_tasks}")
print(f"Workers: {MAX_WORKERS}")
print("="*70)

# Check if there are banks to scrape
if total_tasks == 0:
    print("\n⚠️  WARNING: No banks found for selected provinces!")
    print(f"Available provinces in BPR Syariah data:")
    for prov in sorted(df_banks['Provinsi'].unique()):
        count = len(df_banks[df_banks['Provinsi'] == prov])
        print(f"  - {prov}: {count} banks")
    print("\nPlease update SELECTED_PROVINCES in the configuration cell.")
else:
    print()
    
    # Run scraping
    start_time = time.time()
    scraper = BPRSScraper()
    results = scraper.run(filtered_banks, years, max_workers=MAX_WORKERS)
    elapsed_time = time.time() - start_time
    
    # Process results
    df_results = pd.DataFrame(results)
    
    # Add missing columns
    for col in COLUMN_ORDER:
        if col not in df_results.columns:
            df_results[col] = 0
    
    # Reorder columns
    available_cols = [col for col in COLUMN_ORDER if col in df_results.columns]
    df_results = df_results[available_cols]
    
    # Statistics
    success_count = (df_results['status'] == 'success').sum() if 'status' in df_results.columns else len(df_results)
    failed_count = total_tasks - success_count
    
    print("\n" + "="*70)
    print(" SCRAPING COMPLETED")
    print("="*70)
    print(f"Total Tasks: {total_tasks}")
    print(f"Success: {success_count} ({success_count/total_tasks*100:.1f}%)")
    print(f"Failed: {failed_count} ({failed_count/total_tasks*100:.1f}%)")
    print(f"Elapsed Time: {elapsed_time:.2f} seconds")
    print(f"Average: {elapsed_time/total_tasks:.2f} sec/task")
    print("="*70)
    
    # Remove status column
    df_export = df_results.drop(columns=['status'], errors='ignore')
    
    # Replace 0 with empty string for ratio columns
    ratio_cols = ['KPMM', 'NPF_Neto', 'NPF_Gross', 'ROA', 'BOPO', 'FDR', 'Cash_Ratio', 'NI']
    for col in ratio_cols:
        if col in df_export.columns:
            df_export[col] = df_export[col].replace(0, '')
    
    # Save files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_file = f"{OUTPUT_FILENAME}_{timestamp}.csv"
    df_export.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"\n✓ Saved: {csv_file}")
    
    try:
        excel_file = f"{OUTPUT_FILENAME}_{timestamp}.xlsx"
        df_export.to_excel(excel_file, index=False, engine='openpyxl')
        print(f"✓ Saved: {excel_file}")
    except Exception as e:
        print(f"Excel save failed: {e}")
    
    print("\n✓ Done!")

 STARTING SCRAPING - BPR SYARIAH
Tahun: 2024 - 2024 (1 tahun)
Bulan: Desember (Fixed)
Provinsi: SEMUA
Jumlah Bank: 194
Total Tasks: 194
Workers: 8



Scraping:   0%|          | 0/194 [00:00<?, ?task/s]


 SCRAPING COMPLETED
Total Tasks: 194
Success: 194 (100.0%)
Failed: 0 (0.0%)
Elapsed Time: 408.01 seconds
Average: 2.10 sec/task

✓ Saved: bprs_financial_data_20260107_210556.csv
✓ Saved: bprs_financial_data_20260107_210556.xlsx

✓ Done!


## 9. Preview Results

In [11]:
print("Sample Data (First 5 rows):")
df_export.head()

Sample Data (First 5 rows):


,Tahun,Bulan,Longitude,Latitude,Nama_BPR,Kabupaten_Kota,Provinsi,Kode_Bank,Total_Aset,Piutang_Murabahah,...,Zakat,Laba_Rugi_Bersih,KPMM,NPF_Neto,NPF_Gross,ROA,BOPO,FDR,Cash_Ratio,NI
0,2024,12,None,None,PT. BPRS Rif'atul Ummah,Kab. Bogor,Provinsi Jawa Barat,620068,2.902445e+10,6.896778e+09,...,0.0,-1.710971e+09,36.82,20.74,25.40,-5.89,197.42,104.51,10.2,36.82
1,2024,12,None,None,PT Bank Perekonomian Rakyat Syariah Amanah Insani,Kab. Bekasi,Provinsi Jawa Barat,620066,7.087924e+10,4.837444e+10,...,451398691.0,1.323343e+09,72.94,21.38,23.17,1.62,89.24,103.01,24.25,72.94
2,2024,12,None,None,PT Bank Perekonomian Rakyat Syariah Botani Bin...,Kab. Bogor,Provinsi Jawa Barat,620061,1.421886e+11,7.777593e+10,...,0.0,2.438850e+09,20.36,0.01,2.66,1.75,76.37,95.31,8.52,20.36
3,2024,12,None,None,PT Bank Perekonomian Rakyat Syariah Amanah Rab...,Kab. Bandung,Provinsi Jawa Barat,620002,1.541767e+11,8.946484e+10,...,144000000.0,3.735341e+09,19.85,2.43,4.79,3.29,88.69,75.33,21.36,19.85
4,2024,12,None,None,PT Bank Perekonomian Rakyat Syariah Gaido Indo...,Kab. Cianjur,Provinsi Jawa Barat,620043,3.562391e+10,2.581306e+10,...,0.0,-1.669246e+09,5.7,6.5,7.95,-4.69,133.7,86.62,28.26,5.7


In [26]:
dx = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/FinalBPRS.csv')
dx = dx[dx['Tahun'] == 2024]
dx.describe().T

,count,mean,std,min,25%,50%,75%,max
Tahun,202.0,2.024000e+03,0.000000e+00,2.024000e+03,2.024000e+03,2.024000e+03,2.024000e+03,2.024000e+03
Bulan,202.0,1.200000e+01,0.000000e+00,1.200000e+01,1.200000e+01,1.200000e+01,1.200000e+01,1.200000e+01
Kode_Bank,202.0,6.201029e+05,6.058220e+01,6.200010e+05,6.200502e+05,6.201075e+05,6.201578e+05,6.202030e+05
Total_Aset,175.0,1.468561e+11,2.228084e+11,4.603144e+07,3.962819e+10,7.538476e+10,1.593760e+11,1.896594e+12
Piutang_Murabahah,175.0,6.079275e+10,8.700396e+10,0.000000e+00,1.810946e+10,3.901493e+10,6.596341e+10,8.260732e+11
Piutang_Istishna,175.0,2.158769e+09,1.850425e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.413183e+11
Piutang_Multijasa,175.0,1.366307e+10,3.881061e+10,0.000000e+00,2.672598e+08,1.553621e+09,7.170380e+09,2.406551e+11
Piutang_Qardh,175.0,4.089165e+09,1.934709e+10,0.000000e+00,0.000000e+00,2.622000e+07,9.148073e+08,2.220651e+11
Piutang_Sewa,175.0,6.791353e+07,2.783130e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.931954e+09
Pembiayaan_Mudharabah,175.0,1.452510e+09,4.114874e+09,0.000000e+00,0.000000e+00,0.000000e+00,1.006105e+09,2.863037e+10


In [15]:
print("Summary Statistics (Transposed):")
df_export.describe().T

Summary Statistics (Transposed):


,count,mean,std,min,25%,50%,75%,max
Tahun,194.0,2.024000e+03,0.000000e+00,2.024000e+03,2.024000e+03,2.024000e+03,2.024000e+03,2.024000e+03
Total_Aset,171.0,1.449210e+11,2.230984e+11,4.603144e+07,4.109559e+10,7.538476e+10,1.526587e+11,1.896594e+12
Piutang_Murabahah,171.0,6.148428e+10,8.789385e+10,0.000000e+00,1.793873e+10,3.901493e+10,6.753794e+10,8.260732e+11
Piutang_Istishna,171.0,2.209266e+09,1.871768e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.413183e+11
Piutang_Multijasa,171.0,1.114809e+10,3.045453e+10,0.000000e+00,2.604317e+08,1.473907e+09,6.761351e+09,2.406551e+11
Piutang_Qardh,171.0,3.943928e+09,1.950949e+10,0.000000e+00,0.000000e+00,1.553333e+07,7.802547e+08,2.220651e+11
Piutang_Sewa,171.0,6.945829e+07,2.813815e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.931954e+09
Pembiayaan_Mudharabah,171.0,1.481809e+09,4.158351e+09,0.000000e+00,0.000000e+00,0.000000e+00,1.080090e+09,2.863037e+10
Pembiayaan_Musyarakah,171.0,2.914057e+10,9.562697e+10,0.000000e+00,5.022917e+08,5.595227e+09,2.066812e+10,9.761613e+11
Pembiayaan_Lainnya,171.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


## 10. Data Quality Check

**Check ratio values:**

In [ ]:
ratio_cols = ['KPMM', 'NPF_Neto', 'NPF_Gross', 'ROA', 'BOPO', 'FDR', 'Cash_Ratio', 'NI']

print("✓ Ratio Data Check:")
print("="*50)
for col in ratio_cols:
    if col in df_export.columns:
        # Count non-zero AND non-empty values
        non_zero = ((df_export[col] != 0) & (df_export[col] != '')).sum()
        total = len(df_export)
        pct = (non_zero / total * 100) if total > 0 else 0
        print(f"{col:20s}: {non_zero:3d}/{total:3d} ({pct:5.1f}%) have values")

print("\n✓ Ratio Statistics:")
df_export[ratio_cols].describe()

## 11. Top Banks by Total Aset

In [ ]:
latest_year = df_export['Tahun'].max()
df_latest = df_export[df_export['Tahun'] == latest_year].copy()

print(f"✓ Top 10 BPRS by Total Aset ({latest_year}):")
df_top = df_latest.nlargest(10, 'Total_Aset')[['Nama_BPR', 'Provinsi', 'Total_Aset', 'ROA', 'NPF_Gross']]
df_top

---

## 📝 Notes:

### Key Features:
- ✅ **Khusus BPR Syariah** - Data tahun 2019-2024
- ✅ **Thread-safe** - Menggunakan ThreadPoolExecutor
- ✅ **Progress tracking** - Real-time progress dengan tqdm
- ✅ **Error handling** - Robust error handling
- ✅ **Auto-save** - CSV dan Excel otomatis tersimpan
- ✅ **Table Index 16** - Untuk semua laporan kecuali Informasi Lainnya

### Tips:
- Start dengan `MAX_WORKERS=4-6` untuk stability
- Untuk dataset besar, scrape per provinsi
- Check ratio values dengan cell "Data Quality Check"
- Jika timeout banyak, kurangi `MAX_WORKERS`

### Column Mapping:
Sesuai dengan Dataset Syakhsan.xlsx sheet "BPRS (2019 - 2024)"

### Troubleshooting:
- **Ratio masih 0?** → Cek format table atau keywords
- **Banyak failed?** → Kurangi MAX_WORKERS atau cek koneksi internet
- **Memory error?** → Scrape per provinsi, jangan semua sekaligus